# Source 1: (REST API) — OpenSky Network

Documentation can be found here: `https://openskynetwork.github.io/opensky-api/rest.html`.

- Endpoint: `https://opensky-network.org/api/states/all`.
- We don't want an account, so we use the rate limits for Anonymous users (10 seconds).
- As we focus on Europe, we restrict to a **Europe bounding box** via coordinates (lat 34–72°N, lon −25–50°E).

In [22]:
import requests
import pandas as pd

OPENSKY_URL = "https://opensky-network.org/api/states/all"
EUROPE_BBOX = {"lamin": 34.0, "lomin": -25.0, "lamax": 72.0, "lomax": 50.0}

resp = requests.get(OPENSKY_URL, params=EUROPE_BBOX, timeout=60)
resp.raise_for_status()
data = resp.json()
print("HTTP", resp.status_code)
print("top-level keys:", list(data.keys()))
print("snapshot time (unix):", data["time"])
print("aircraft in this snapshot:", len(data["states"]))

HTTP 200
top-level keys: ['time', 'states']
snapshot time (unix): 1782580633
aircraft in this snapshot: 3456


## Response
The API returns two keys: `time` (the snapshot's timestamp) and `states` (a list of aircraft).

In [23]:
import json
print("one raw state vector:")
print(json.dumps(data["states"][0]))
print("length:", len(data["states"][0]))

one raw state vector:
["39de4f", "TVF8965 ", "France", 1782580632, 1782580633, 4.0281, 47.4023, 8945.88, false, 233.16, 325.61, -11.38, null, 9517.38, "1000", false, 0]
length: 17


## Field reference (this is just copied from the API-reference)

| idx | field | meaning | unit | use for us |
|----:|-------|---------|------|------------|
| 0 | `icao24` | unique aircraft id (hex) | — | identify an aircraft |
| 1 | `callsign` | flight callsign | — | **commercial filter** (first 3 letters = airline ICAO) |
| 2 | `origin_country` | country of registration/operator | — | **dominance by country** |
| 3 | `time_position` | last position update | unix s | freshness |
| 4 | `last_contact` | last any update | unix s | freshness |
| 5 | `longitude` | position | deg | **geography / spatial join** |
| 6 | `latitude` | position | deg | **geography / spatial join** |
| 7 | `baro_altitude` | barometric altitude | m | altitude |
| 8 | `on_ground` | parked/taxiing? | bool | filter out ground |
| 9 | `velocity` | ground speed | m/s | speed |
| 10 | `true_track` | heading (0=N) | deg | direction |
| 11 | `vertical_rate` | climb(+)/descent(−) | m/s | **busiest-airports (climb/descent)** |
| 12 | `sensors` | receiving sensors | — | (always null here — drop) |
| 13 | `geo_altitude` | geometric altitude | m | altitude |
| 14 | `squawk` | transponder code | — | optional |
| 15 | `spi` | special purpose indicator | bool | optional |
| 16 | `position_source` | source of position | enum | optional |

For some cleaning, let's put this into a DataFrame:

In [24]:
COLS = [
    "icao24", "callsign", "origin_country", "time_position", "last_contact",
    "longitude", "latitude", "baro_altitude", "on_ground", "velocity",
    "true_track", "vertical_rate", "sensors", "geo_altitude", "squawk",
    "spi", "position_source",
]
df = pd.DataFrame(data["states"], columns=COLS)
df["callsign"] = df["callsign"].str.strip()
df.shape

(3456, 17)

## Data quality

Is ok, structured (as it's an API), but definitely will need some cleaning. Below are a few points which have been identified and have to be dealt with later.

`Sensors` is always empty, so later in the pipeline we will just drop this. `Squawk` too (sometimes null, not needed). Same with `spi` and `position source`.

Also: `geo_altitude` / `baro_altitude` / `vertical_rate` are ~8% null, and there are clear outliers (e.g. `velocity` up to 858 m/s ≈ 3.000 km/h, negative altitudes below sea level). These are exactly the fields the busiest-airports question (Q3) relies on, so they need handling (null filtering + outlier removal) before the altitude / climb-descent analysis.

In [4]:
(df.isna().mean() * 100).round(1)

icao24               0.0
callsign             0.0
origin_country       0.0
time_position        0.0
last_contact         0.0
longitude            0.0
latitude             0.0
baro_altitude        7.8
on_ground            0.0
velocity             0.1
true_track           0.0
vertical_rate        8.1
sensors            100.0
geo_altitude         9.0
squawk              15.6
spi                  0.0
position_source      0.0
dtype: float64

We only want flying aircraft, so we need to filter out the `on_ground` ones as well (`on_ground == False`). Furthermore, we can only use aircrafts with a valid callsign, so we also filter here (`callsign`).

In [28]:
print(df["on_ground"].value_counts())
air = df[(~df["on_ground"]) & (df["callsign"] != "")].copy()
print("\nairborne with a callsign:", len(air))

on_ground
False    3168
True      288
Name: count, dtype: int64

airborne with a callsign: 3157


## Interesting fields for our questions (This is an excerpt at a certain point in time. It's not guaranteed that the final questions are exactly 1:1.)

**Q1 Dominance — `origin_country`** (operator country) and the **callsign prefix** (airline, via the scraped codes).

In [29]:
print("top operator countries:")
print(air["origin_country"].value_counts().head(10))
print("\nsample callsigns (prefix = airline ICAO, e.g. TVF=Transavia, EZY=easyJet, UAL=United):")
print(air["callsign"].head(12).tolist())
air["airline_icao"] = air["callsign"].str[:3]
print("\ntop callsign prefixes:")
print(air["airline_icao"].value_counts().head(10))

top operator countries:
origin_country
United Kingdom                429
Ireland                       315
Germany                       254
Turkey                        229
Malta                         175
France                        174
Spain                         155
Poland                        127
Austria                       115
Kingdom of the Netherlands     90
Name: count, dtype: int64

sample callsigns (prefix = airline ICAO, e.g. TVF=Transavia, EZY=easyJet, UAL=United):
['TVF8965', 'TVF41HB', 'UAL933', 'TVF65YB', 'TVF8230', 'TVF38EU', 'DMADH', 'FJMIV', 'TVF15GP', 'EZY24DC', 'TVF8602', 'TVF93NR']

top callsign prefixes:
airline_icao
RYR    391
EXS    105
EZY    102
THY     92
SAS     81
DLH     76
AFR     72
VLG     68
BAW     68
KLM     64
Name: count, dtype: int64


**Q3 Busiest airports — `latitude`/`longitude` + `vertical_rate` + altitude.** Aircraft that are low and climbing have just departed; low and descending are arriving. We'll spatial-join these to the nearest airport (source 3).

In [30]:
print("altitude / speed / vertical-rate ranges (airborne):")
print(air[["geo_altitude", "velocity", "vertical_rate"]].describe().round(1))
# rough climb/descent split (positive/negative vertical rate, low altitude)
low = air[air["geo_altitude"] < 3000]
print("\nlow-altitude aircraft (<3000m):", len(low))
print("  climbing (vr>1):", (low["vertical_rate"] > 1).sum())
print("  descending (vr<-1):", (low["vertical_rate"] < -1).sum())

altitude / speed / vertical-rate ranges (airborne):
       geo_altitude  velocity  vertical_rate
count        3126.0    3157.0         3154.0
mean         8520.0     196.4           -0.1
std          4230.5      67.3            5.1
min          -701.0       0.0          -24.4
25%          4998.7     166.8           -0.3
50%         10919.5     224.7            0.0
75%         11765.3     238.1            0.3
max         14363.7     858.2           45.2

low-altitude aircraft (<3000m): 614
  climbing (vr>1): 117
  descending (vr<-1): 297
